# RSNA Knee MRI — 管线验证

> **目标**：在启动完整 5 折训练之前，验证数据加载 → 增强 → 模型 forward → 损失反向传播全流程正常工作。
>
> **通过标准**：
> - 数据形状正确、无 NaN/Inf
> - 增强可视化合理（不破坏解剖结构）
> - 单 batch 能过拟合（loss < 0.1）
> - 3 epoch 快速训练 macro AUC > 0.52

## 0. 导入与设置

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import yaml
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.data.dataset import KneeSliceDataset
from src.data.transforms import build_transforms, apply_transform
from src.models.classifier import KneeClassifier2D
from src.losses import build_loss
from src.metrics import compute_macro_auc

SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

TARGET_COLS = [
    'ACL', 'MCL',
    'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA',
    'Effusion', 'Synovitis', "Baker's",
    'Contusion', 'Fracture'
]

## 1. 加载配置

In [ ]:
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'a_convnextv2_tiny_2.5d_mini.yaml'

with open(CONFIG_PATH, encoding='utf-8') as f:
    config = yaml.safe_load(f)

model_cfg = config['model']
data_cfg = config['data']
train_cfg = config['train']
aug_cfg = config.get('augmentation', {})
paths_cfg = config['paths']

print('=== 关键配置一览 ===')
print(f'  arch:              {model_cfg["arch"]}')
print(f'  pretrained:        {model_cfg["pretrained"]} ({model_cfg["pretrained_source"]})')
print(f'  image_size:        {data_cfg["image_size"]}')
print(f'  in_channels:       {data_cfg["in_channels"]}')
print(f'  batch_size:        {train_cfg["batch_size"]}')
print(f'  backbone_lr:       {config["optimizer"]["backbone_lr"]}')
print(f'  head_lr:           {config["optimizer"]["head_lr"]}')
print(f'  dropout:           {model_cfg["dropout"]}')
print(f'  label_smoothing:   {config["loss"]["label_smoothing"]}')

## 2. 加载数据

In [ ]:
metadata_path = PROJECT_ROOT / paths_cfg['metadata_csv']
labels_path = PROJECT_ROOT / paths_cfg['train_csv']
npy_root = PROJECT_ROOT / paths_cfg['npy_root']

metadata_df = pd.read_csv(metadata_path)
labels_df = pd.read_csv(labels_path)

for col in TARGET_COLS:
    labels_df[col] = pd.to_numeric(labels_df[col], errors='coerce').fillna(0).astype(int)

print(f'切片索引行数:   {len(metadata_df)}')
print(f'标签 exam 数:   {len(labels_df)}')
print(f'npy 目录存在:   {npy_root.exists()}')

label_counts = labels_df[TARGET_COLS].sum()
print(f'\n=== 标签分布 (n={len(labels_df)} exams) ===')
for col in TARGET_COLS:
    pct = (label_counts[col] / len(labels_df) * 100)
    print(f'  {col:20s}: n={int(label_counts[col]):4d}  ({pct:.2f}%)')

studies_in_meta = set(metadata_df['StudyInstanceUID'].unique())
studies_in_labels = set(labels_df['StudyInstanceUID'].unique())
missing = studies_in_meta - studies_in_labels
print(f'\nmetadata 中有但 labels 中没有: {len(missing)} studies')

## 3. 可视化：原始 vs 增强切片

In [ ]:
image_size = data_cfg['image_size']
in_channels = data_cfg['in_channels']

train_transform = build_transforms(aug_cfg, image_size, is_train=True)
valid_transform = build_transforms(aug_cfg, image_size, is_train=False)

print(f'训练管道 ({len(train_transform)} 步):')
for t in train_transform:
    print(f'  - {t.__class__.__name__}')
print(f'\n验证管道 ({len(valid_transform)} 步):')
for t in valid_transform:
    print(f'  - {t.__class__.__name__}')

In [ ]:
vis_ds = KneeSliceDataset(
    metadata_df, labels_df,
    npy_root=str(npy_root),
    image_size=image_size, in_channels=in_channels,
    slice_offset=data_cfg.get('slice_offset', 1),
    is_train=True,
)

samples = []
for idx in range(len(vis_ds)):
    rec = vis_ds[idx]
    labels = rec['labels'].numpy()
    if len(samples) == 0 and labels.sum() == 0:
        samples.append(idx)
    elif len(samples) == 1 and labels[TARGET_COLS.index('Effusion')] == 1:
        samples.append(idx)
    elif len(samples) == 2 and labels[TARGET_COLS.index('ACL')] == 1:
        samples.append(idx)
        break

for i, idx in enumerate(samples):
    rec = vis_ds[idx]
    active = [TARGET_COLS[j] for j in range(12) if rec['labels'][j] == 1]
    label_str = ', '.join(active) if active else '正常'
    print(f'样本 {i}: label={label_str}')

In [ ]:
def show_augmentation(axes, sample_idx, n_variants=5):
    rec = vis_ds[sample_idx]
    image_np = rec['image'].numpy()
    mid_ch = 1 if in_channels == 3 else 0

    valid = apply_transform(image_np, valid_transform)
    axes[0].imshow(valid[mid_ch], cmap='gray')
    axes[0].set_title('Valid (resize only)', fontsize=9)
    axes[0].axis('off')

    for j in range(n_variants):
        aug = apply_transform(image_np, train_transform)
        axes[j + 1].imshow(aug[mid_ch], cmap='gray')
        axes[j + 1].set_title(f'Aug #{j+1}', fontsize=9)
        axes[j + 1].axis('off')


for sample_idx in samples:
    rec = vis_ds[sample_idx]
    active = [TARGET_COLS[j] for j in range(12) if rec['labels'][j] == 1]
    label_str = ', '.join(active) if active else '正常'

    fig, axes = plt.subplots(1, 6, figsize=(14, 2.5))
    show_augmentation(axes, sample_idx, n_variants=5)
    fig.suptitle(f'Label: {label_str}', fontsize=11, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
print('=== 增强肉眼检查 ===')
print('  □ 水平翻转后解剖结构不混淆 (sagittal 面翻转合法)')
print('  □ 旋转/平移不超出图像边界 (rotate=±10°, shift=5%)')
print('  □ 亮度/对比度/Gamma 变化保持组织可辨识')
print('  □ CoarseDropout 遮挡块 ≤32px, 不覆盖整个结构')
print('  □ 3 通道统一施加空间变换, 切片间对齐保持')

## 4. 验证 DataLoader

In [ ]:
train_ds = KneeSliceDataset(
    metadata_df, labels_df,
    npy_root=str(npy_root),
    image_size=image_size, in_channels=in_channels,
    slice_offset=data_cfg.get('slice_offset', 1),
    is_train=True, transform=train_transform,
)
print(f'训练集样本数: {len(train_ds)}')

In [ ]:
batch_size = min(train_cfg['batch_size'], 8)
loader = DataLoader(
    train_ds, batch_size=batch_size, shuffle=True,
    num_workers=0, pin_memory=False,
)

t0 = time.time()
batch = next(iter(loader))
load_time = time.time() - t0

images = batch['image']
labels = batch['labels']

print(f'=== Batch 验证 ===')
print(f'  image.shape:     {images.shape}')
print(f'  labels.shape:    {labels.shape}')
print(f'  image min/max:   {images.min():.4f} / {images.max():.4f}')
print(f'  image mean/std:  {images.mean():.4f} / {images.std():.4f}')
print(f'  加载耗时:         {load_time*1000:.0f} ms')

t0 = time.time()
for i, b in enumerate(loader):
    if i >= 9:
        break
avg_time = (time.time() - t0) / 10
print(f'  平均耗时:         {avg_time*1000:.0f} ms/batch')

has_nan = torch.isnan(images).any().item()
has_inf = torch.isinf(images).any().item()
print(f'  NaN: {"❌" if has_nan else "✅"}  Inf: {"❌" if has_inf else "✅"}')

## 5. 模型 Forward Pass 验证

In [ ]:
model = KneeClassifier2D(
    arch=model_cfg['arch'],
    pretrained=model_cfg['pretrained'],
    in_channels=model_cfg['in_channels'],
    num_classes=model_cfg['num_classes'],
    dropout=model_cfg['dropout'],
    drop_path_rate=model_cfg['drop_path_rate'],
).to(DEVICE)

n_backbone = sum(p.numel() for p in model.backbone.parameters())
n_head = sum(p.numel() for p in model.head.parameters())
print(f'backbone: {n_backbone/1e6:.1f}M  head: {n_head/1e6:.3f}M  total: {(n_backbone+n_head)/1e6:.1f}M')

In [ ]:
model.eval()
with torch.no_grad():
    images_gpu = images.to(DEVICE, memory_format=torch.channels_last)
    logits = model(images_gpu)

print(f'input:  {images_gpu.shape}')
print(f'output: {logits.shape}')
print(f'logits: [{logits.min():.4f}, {logits.max():.4f}]  mean={logits.mean():.4f}')

if logits.abs().max() < 10:
    print('✅ logits 范围合理')
else:
    print('⚠️  logits 过大')

if DEVICE == 'cuda':
    print(f'GPU 显存: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB')

## 6. 单 Batch 过拟合测试 ⭐

> 固定 1 个 batch, 大学习率训练到 loss 接近 0。通过 = 模型/损失/优化器配合正常。

In [ ]:
overfit_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False, num_workers=0)
overfit_batch = next(iter(overfit_loader))
overfit_images = overfit_batch['image'].to(DEVICE, memory_format=torch.channels_last)
overfit_labels = overfit_batch['labels'].to(DEVICE)

model = KneeClassifier2D(
    arch=model_cfg['arch'], pretrained=model_cfg['pretrained'],
    in_channels=model_cfg['in_channels'], num_classes=model_cfg['num_classes'],
    dropout=model_cfg['dropout'], drop_path_rate=0.0,
).to(DEVICE)
model.train()

criterion = build_loss(config['loss']['name'], label_smoothing=config['loss']['label_smoothing'])
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

print(f'batch={len(overfit_images)}, lr=1e-3, label_smoothing={config["loss"]["label_smoothing"]}')

In [ ]:
n_steps = 150
loss_history = []

for step in range(n_steps):
    optimizer.zero_grad()
    logits = model(overfit_images)
    loss = criterion(logits, overfit_labels)
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())

    if step % 25 == 0 or step == n_steps - 1:
        with torch.no_grad():
            acc = ((torch.sigmoid(logits) > 0.5) == overfit_labels).float().mean().item()
        print(f'  step {step:3d}: loss={loss.item():.4f}  acc={acc:.4f}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(loss_history, color='#2c3e50', linewidth=0.8)
ax.axhline(0.05, color='#e74c3c', linestyle='--', alpha=0.5, label='target < 0.05')
ax.set_xlabel('Step'); ax.set_ylabel('Loss')
ax.set_title('Single-Batch Overfitting Test')
ax.legend()
plt.tight_layout(); plt.show()

final_loss = loss_history[-1]
if final_loss < 0.05:
    print(f'✅ 过拟合测试通过! final_loss={final_loss:.4f}')
elif final_loss < 0.1:
    print(f'⚠️  基本通过, final_loss={final_loss:.4f}')
else:
    print(f'❌ 未通过! final_loss={final_loss:.4f} > 0.1')

## 7. 快速训练验证 (3 epochs, 单折)

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
study_labels = labels_df.set_index('StudyInstanceUID')[TARGET_COLS].max(axis=1)

train_idx, valid_idx = next(sgkf.split(
    labels_df, study_labels, groups=labels_df['StudyInstanceUID']
))

train_studies = labels_df.iloc[train_idx]['StudyInstanceUID'].unique()
valid_studies = labels_df.iloc[valid_idx]['StudyInstanceUID'].unique()

print(f'Train studies: {len(train_studies)}')
print(f'Valid studies: {len(valid_studies)}')
print(f'Overlap: {len(set(train_studies) & set(valid_studies))} (should be 0)')

In [ ]:
train_meta = metadata_df[metadata_df['StudyInstanceUID'].isin(train_studies)]
valid_meta = metadata_df[metadata_df['StudyInstanceUID'].isin(valid_studies)]

quick_train_ds = KneeSliceDataset(
    train_meta, labels_df, npy_root=str(npy_root),
    image_size=image_size, in_channels=in_channels,
    slice_offset=data_cfg.get('slice_offset', 1),
    is_train=True, transform=train_transform,
)
quick_valid_ds = KneeSliceDataset(
    valid_meta, labels_df, npy_root=str(npy_root),
    image_size=image_size, in_channels=in_channels,
    slice_offset=data_cfg.get('slice_offset', 1),
    is_train=False, transform=valid_transform,
)

quick_train_loader = DataLoader(quick_train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
quick_valid_loader = DataLoader(quick_valid_ds, batch_size=batch_size, shuffle=False, num_workers=0)

print(f'Train batches: {len(quick_train_loader)}, Valid batches: {len(quick_valid_loader)}')

In [ ]:
model = KneeClassifier2D(
    arch=model_cfg['arch'], pretrained=model_cfg['pretrained'],
    in_channels=model_cfg['in_channels'], num_classes=model_cfg['num_classes'],
    dropout=model_cfg['dropout'], drop_path_rate=model_cfg['drop_path_rate'],
).to(DEVICE)

criterion = build_loss(config['loss']['name'], label_smoothing=config['loss']['label_smoothing'])
optimizer = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': config['optimizer']['backbone_lr']},
    {'params': model.head.parameters(), 'lr': config['optimizer']['head_lr']},
], weight_decay=config['optimizer']['weight_decay'])

use_amp = train_cfg['mixed_precision']
scaler = torch.cuda.amp.GradScaler() if use_amp else None
grad_clip = train_cfg['gradient_clip_norm']

print(f'AMP: {"fp16" if use_amp else "fp32"}, grad_clip: {grad_clip}')

In [ ]:
quick_epochs = 3
history = {'train_loss': [], 'val_loss': [], 'val_auc': []}

for epoch in range(quick_epochs):
    model.train()
    train_loss = 0.0
    t0 = time.time()
    for batch in quick_train_loader:
        imgs = batch['image'].to(DEVICE, memory_format=torch.channels_last)
        lbls = batch['labels'].to(DEVICE)

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(imgs)
            loss = criterion(logits, lbls)

        optimizer.zero_grad()
        if scaler:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        train_loss += loss.item()
    train_loss /= len(quick_train_loader)
    train_time = time.time() - t0

    model.eval()
    val_loss = 0.0
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in quick_valid_loader:
            imgs = batch['image'].to(DEVICE, memory_format=torch.channels_last)
            lbls = batch['labels'].to(DEVICE)
            logits = model(imgs)
            val_loss += criterion(logits, lbls).item()
            all_logits.append(logits.cpu().numpy())
            all_labels.append(lbls.cpu().numpy())

    val_loss /= len(quick_valid_loader)
    val_auc = compute_macro_auc(np.concatenate(all_labels), np.concatenate(all_logits))

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)

    print(f'Epoch {epoch+1}: train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_auc={val_auc:.4f}  ({train_time:.0f}s)')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs_range = range(1, quick_epochs + 1)

ax1.plot(epochs_range, history['train_loss'], 'o-', color='#2c3e50', label='Train Loss')
ax1.plot(epochs_range, history['val_loss'], 'o-', color='#e74c3c', label='Valid Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Loss'); ax1.legend(); ax1.set_xticks(epochs_range)

ax2.plot(epochs_range, history['val_auc'], 'o-', color='#27ae60', linewidth=2, markersize=8)
ax2.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Random (0.5)')
ax2.axhline(0.52, color='#e74c3c', linestyle='--', alpha=0.5, label='Threshold (0.52)')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Macro AUC')
ax2.set_title('Validation Macro AUC'); ax2.legend(); ax2.set_xticks(epochs_range)

plt.tight_layout(); plt.show()

final_auc = history['val_auc'][-1]
if final_auc > 0.55:
    print(f'✅ AUC={final_auc:.4f} > 0.55, 管线完全正常')
elif final_auc > 0.52:
    print(f'⚠️  AUC={final_auc:.4f} > 0.52, 优于随机, 可启动完整训练')
else:
    print(f'❌ AUC={final_auc:.4f} ≈ 随机, 需排查')

## 8. 结论与后续步骤

In [ ]:
print('=== 管线验证检查清单 ===')
print()
print('  □ 数据加载正常, shape 正确              (§2 + §4)')
print('  □ 增强可视化合理, 无解剖结构破坏        (§3)')
print('  □ 无 NaN/Inf 泄漏                       (§4)')
print('  □ DataLoader 性能可接受                  (§4)')
print('  □ 模型 Forward Pass 正常                 (§5)')
print('  □ GPU 显存未超限                          (§5)')
print('  □ 单 batch 过拟合通过 (loss < 0.1)       (§6)')
print('  □ 快速训练 macro AUC > 0.52              (§7)')
print()
print('---')
print()
print('全部 ✓ → 运行完整训练:')
print('  python src/train.py --config configs/a_convnextv2_tiny_2.5d_mini.yaml')
print()
print('升级到 ConvNeXtV2-Small: 改 model.arch = convnextv2_small')
print()
print('后续 notebook:')
print('  04_gradcam_analysis.ipynb  — Grad-CAM 热力图分析')
print('  05_error_analysis.ipynb    — 错误案例 + 难分类别定位')